<a href="https://colab.research.google.com/github/npeyton/ml_materials/blob/main/nielsen_neural_networks_chap_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#### Libraries
# Standard library
import random

# Third-party libraries
import numpy as np

Fully matrix-based approach to backpropagation over a mini-batch Our implementation of stochastic gradient descent loops over training examples in a mini-batch. It's possible to modify the backpropagation algorithm so that it computes the gradients for all training examples in a mini-batch simultaneously. The idea is that instead of beginning with a single input vector, x
, we can begin with a matrix X=[x1x2…xm]
 whose columns are the vectors in the mini-batch. We forward-propagate by multiplying by the weight matrices, adding a suitable matrix for the bias terms, and applying the sigmoid function everywhere. We backpropagate along similar lines. Explicitly write out pseudocode for this approach to the backpropagation algorithm. Modify network.py so that it uses this fully matrix-based approach. The advantage of this approach is that it takes full advantage of modern libraries for linear algebra. As a result it can be quite a bit faster than looping over the mini-batch. (On my laptop, for example, the speedup is about a factor of two when run on MNIST classification problems like those we considered in the last chapter.) In practice, all serious libraries for backpropagation use this fully matrix-based approach or some variant.

Solution:
First I need to Stack the inputs so that they are in a matrix as opposed to how they were before in separate columns.

Then I can to the same Forward Pass -> Calculate Error At each layer -> move backward for each layer -> calculate the gradients at each layer -> update the weights and biases

In [6]:
#getting the code from the first lesson.
!git clone https://github.com/unexploredtest/neural-networks-and-deep-learning.git
%cd neural-networks-and-deep-learning/src

Cloning into 'neural-networks-and-deep-learning'...
remote: Enumerating objects: 1227, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 1227 (delta 21), reused 17 (delta 17), pack-reused 1175 (from 3)
Receiving objects: 100% (1227/1227), 20.45 MiB | 4.45 MiB/s, done.
Resolving deltas: 100% (606/606), done.
/content/neural-networks-and-deep-learning/src


In [7]:
import mnist_loader

In [8]:
training_data, validation_data, test_data = mnist_loader.load_data_wrapper()
from network import Network, sigmoid, sigmoid_prime

/content/neural-networks-and-deep-learning/src/network.py:134: SyntaxWarning: invalid escape sequence '\p'
  """Return the vector of partial derivatives \partial C_x /


In [9]:
class MatrixNetwork(Network):

    def update_mini_batch(self, mini_batch, eta):
        # Step 1: stack the batch into matrices
        X = np.hstack([x for x, y in mini_batch])
        Y = np.hstack([y for x, y in mini_batch])
        nabla_b, nabla_w = self.backprop_matrix(X, Y)
        # Step 6: update
        m = len(mini_batch)
        self.weights = [w - (eta/m)*nw for w, nw in zip(self.weights, nabla_w)]
        self.biases  = [b - (eta/m)*nb for b, nb in zip(self.biases, nabla_b)]

    def backprop_matrix(self, X, Y):
        # Step 2: forward pass on all columns at once
        activation = X
        activations = [X]
        zs = []
        for b, w in zip(self.biases, self.weights):
            z = np.dot(w, activation) + b
            zs.append(z)
            activation = sigmoid(z)
            activations.append(activation)

        nabla_b = [None] * len(self.biases)
        nabla_w = [None] * len(self.weights)

        # Step 3: output error
        delta = self.cost_derivative(activations[-1], Y) * sigmoid_prime(zs[-1])
        # Step 5 for the last layer
        nabla_b[-1] = np.sum(delta, axis=1, keepdims=True)
        nabla_w[-1] = np.dot(delta, activations[-2].T)

        # Step 4 and 5 for earlier layers
        for l in range(2, self.num_layers):
            delta = np.dot(self.weights[-l+1].T, delta) * sigmoid_prime(zs[-l])
            nabla_b[-l] = np.sum(delta, axis=1, keepdims=True)
            nabla_w[-l] = np.dot(delta, activations[-l-1].T)

        return nabla_b, nabla_w

In [10]:
import time, mnist_loader, network

training_data, validation_data, test_data = mnist_loader.load_data_wrapper()
training_data, test_data = list(training_data), list(test_data)


In [12]:
#timing the run of the old back prop algorithm over 5 epochs.
start = time.time()
network.Network([784, 30, 10]).SGD(training_data, 5, 10, 3.0, test_data=test_data)
print("Loop version:", round(time.time() - start), "seconds")

Epoch 0: 9163 / 10000, took 4.03 seconds
Epoch 1: 9256 / 10000, took 4.50 seconds
Epoch 2: 9283 / 10000, took 4.24 seconds
Epoch 3: 9374 / 10000, took 4.09 seconds
Epoch 4: 9384 / 10000, took 4.60 seconds
Loop version: 22 seconds


In [13]:
# Timing the run of the Matrix Network over 5 epochs
start = time.time()
MatrixNetwork([784, 30, 10]).SGD(training_data, 5, 10, 3.0, test_data=test_data)
print("Matrix version:", round(time.time() - start), "seconds")

Epoch 0: 8193 / 10000, took 0.94 seconds
Epoch 1: 8330 / 10000, took 0.95 seconds
Epoch 2: 8386 / 10000, took 0.94 seconds
Epoch 3: 8469 / 10000, took 0.99 seconds
Epoch 4: 8473 / 10000, took 0.96 seconds
Matrix version: 6 seconds
